In [ ]:
import os
import subprocess
import pandas as pd
import gwaslab as gl

from pathlib import Path
from plots.plotting import plot_mqq_mb, miami
from src.run_metal_plink2 import run_metal

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Run METAL

In [ ]:
sex = 'female'
region = 'PAR'
label = 'GP2_EUR_AJ' # GP2_EUR_AJ or GP2_FIGS_EUR_AJ

# GP2
file1 = f'{wd}/working/R{rel}_prep/XWAS/{sex}/{region}/XWAS_AJ_{region}_{sex}.PHENO.glm.logistic.hybrid'
file2 = f'{wd}/working/R{rel}_prep/XWAS/{sex}/{region}/XWAS_EUR_{region}_{sex}.PHENO.glm.logistic.hybrid'

# FIGS
file3 = f'{wd}/working/FIGS/AJ/{sex}/gwas_results/XWAS_AJ_{region}_{sex}.PHENO.glm.logistic.hybrid'
file4 = f'{wd}/working/FIGS/EUR/{sex}/gwas_results/XWAS_EUR_{region}_{sex}.PHENO.glm.logistic.hybrid'

In [ ]:
# plink2 --split-par writes PAR1/PAR2 as the chromosome code, but everything downstream
# expects XY (plink code 25): the LocusZoom fix below rewrites XY -> X, and the PAR plots
# in plots/plotting.py filter on chr 25. Recode #CHROM before the meta-analysis.

prep_dir = f'{wd}/working/R{rel}_prep/XWAS/meta/prepped_PAR'
os.makedirs(prep_dir, exist_ok=True)


def recode_par_chrom(path, out_dir=prep_dir):
    """Return a path with PAR1/PAR2 recoded to XY, or the original path if it needs no change."""
    if not os.path.exists(path):
        print(f'not found, left as-is: {path}')
        return path
    with open(path) as fh:
        fh.readline()                                  # header
        first_chrom = fh.readline().split('\t', 1)[0]  # first data row's #CHROM
    if not first_chrom.startswith('PAR'):
        return path
    out = f'{out_dir}/{Path(path).name}.XY'
    subprocess.run(
        'awk \'BEGIN{FS=OFS="\\t"} NR==1{print; next} '
        '{if($1=="PAR1"||$1=="PAR2")$1="XY"; print}\' '
        f"'{path}' > '{out}'",
        shell=True, check=True
    )
    print(f'recoded PAR1/PAR2 -> XY: {out}')
    return out


meta_file1 = recode_par_chrom(file1)
meta_file2 = recode_par_chrom(file2)

if label == "GP2_FIGS_EUR_AJ":
    meta_file3 = recode_par_chrom(file3)
    meta_file4 = recode_par_chrom(file4)

In [ ]:
# FIGS cohorts are included only when label is GP2_FIGS_EUR_AJ

if label == "GP2_FIGS_EUR_AJ":
    meta_files = [meta_file1, meta_file2, meta_file3, meta_file4]
else:
    meta_files = [meta_file1, meta_file2]

res = run_metal(
    metal_input=meta_files,
    metal_outdir=f"{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}",
    work_dir=f"{wd}/metal_tmp",
    marker_id="chrpos",
    effect_allele='ALT',
    metal_prefix=f"{label}_{region}_{sex}"
)

In [ ]:
metal_path = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/{label}_{region}_{sex}_annotated.tsv.gz'

metal = pd.read_csv(metal_path, sep = '\t', compression='gzip')

# HetPVal = Cochran's heterogeneity p-value, HetISq = I^2 (%)
metal = metal[metal.HetISq < 80]
display(metal[metal.Pvalue == metal.Pvalue.min()])
display(metal[metal.Pvalue < 5e-8])

In [ ]:
# LocusZoom cannot read the XY chromosome code -- write a copy with XY relabelled to X
metal_fixed_path = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/{label}_{region}_{sex}_FIXED.tsv.gz'

if region == 'PAR':
    metal.replace({'XY': 'X'}, inplace = True)
    metal.to_csv(metal_fixed_path, compression = 'gzip', sep = '\t', index = False)

### Merge METAL Results with LocusZoom rsIDs

In [ ]:
# Load in from LocusZoom
locuszoom = pd.read_csv(f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/locuszoom/{label}_{region}_{sex}.gz', sep = '\t', compression = 'gzip')
display(locuszoom)

In [ ]:
# Load in METAL and match the LocusZoom column names
metal = pd.read_csv(metal_fixed_path, sep = '\t', compression='gzip')

metal['#chrom'] = metal['CHR']
metal['pos'] = metal['POS'].astype(int)
metal['ref'] = metal['REF']
metal['alt'] = metal['ALT']

display(metal)

In [ ]:
# Merge and filter on heterogeneity + number of contributing studies
combined = locuszoom.merge(metal, on = ['#chrom', 'pos', 'ref', 'alt'], how = 'inner')

if label == 'GP2_EUR_AJ':
    combined = combined[(combined.HetISq < 80) & (combined.HetDf >= 1)]
else:
    combined = combined[(combined.HetISq < 80) & (combined.HetDf >= 2)]
display(combined)

merged_path = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/locuszoom/merged/{label}_{region}_{sex}_filtered.tsv.gz'
combined.to_csv(merged_path, sep = '\t', compression = 'gzip', index = False)

### GWASlab Lead Variants

In [ ]:
sex = 'combined'
region = 'PAR'
label = 'GP2_EUR_AJ'

merged_path = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/locuszoom/merged/{label}_{region}_{sex}_filtered.tsv.gz'
sumstats = gl.Sumstats(merged_path,
             rsid="rsid",
             chrom="#chrom",
             pos="pos",
             ea="alt",
             nea="ref",
             eaf="alt_allele_freq",
             p="Pvalue", # Pvalue or PvalueARE
             n="TotalSampleSize",
             beta='beta',
             se='stderr_beta',
             sep="\t")

sumstats.basic_check()

In [ ]:
sumstats.get_lead(sig_level=5e-8, anno=True, build="38")

In [ ]:
sumstats.plot_qq(stratified = True)

### Load Filtered METAL Results

In [ ]:
# Builds the merged-file path from wd/rel, so it stays with the notebook rather
# than plots/plotting.py
def load_sumstats(sex, region, label):
    """Load one filtered METAL file through GWASlab and return a plot-ready frame."""
    merged_path = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/locuszoom/merged/{label}_{region}_{sex}_filtered.tsv.gz'
    s = gl.Sumstats(merged_path,
                    rsid="rsid",
                    chrom="#chrom",
                    pos="pos",
                    ea="alt",
                    nea="ref",
                    eaf="alt_allele_freq",
                    p="Pvalue",  # Pvalue or PvalueARE
                    n="TotalSampleSize",
                    beta="beta",
                    se="stderr_beta",
                    sep="\t")
    s.basic_check()
    d = s.data.dropna(subset=["CHR", "POS", "P"])
    d = d[d["P"] > 0]                      # guard against log10(0)
    return d.sort_values("POS").reset_index(drop=True)

### Non-PAR Plots

In [ ]:
label = 'GP2_EUR_AJ'
region = 'non_PAR'
sex = 'combined'
sig = 5e-8
nominal = 1e-6   # set to None to drop the dotted line and the extra labels
win_kb = 500

d = load_sumstats(sex, region, label)
plot_mqq_mb(d, sig=sig, win_kb=win_kb, out=f"{wd}/{label}_{region}_{sex}.png")
plot_mqq_mb(d, sig=sig, nominal=nominal, win_kb=win_kb, out=f"{wd}/{label}_{region}_{sex}_nominal.png")

In [ ]:
# Miami plot: males on top, females mirrored below
sexes = ['male', 'female']

top    = load_sumstats(sexes[0], region, label)
bottom = load_sumstats(sexes[1], region, label)

miami(top, bottom,
      titles=[s.capitalize() for s in sexes],
      sig=sig, nominal=nominal, win_kb=win_kb,
      out=f"{wd}/miami_{label}_{region}_{sexes[0]}_vs_{sexes[1]}_nominal.png")

### PAR Plots

In [ ]:
region = 'PAR'
sex = 'combined'
win_kb = 100   # decreased for smaller PAR region

d = load_sumstats(sex, region, label)
plot_mqq_mb(d, sig=sig, win_kb=win_kb, out=f"{wd}/{label}_{region}_{sex}.png", squeeze_gaps=True)
plot_mqq_mb(d, sig=sig, nominal=nominal, win_kb=win_kb,
            out=f"{wd}/{label}_{region}_{sex}_nominal.png", squeeze_gaps=True)

In [ ]:
# Miami plot: males on top, females mirrored below
sexes = ['male', 'female']

top    = load_sumstats(sexes[0], region, label)
bottom = load_sumstats(sexes[1], region, label)

miami(top, bottom,
      titles=[s.capitalize() for s in sexes],
      sig=sig, nominal=nominal, win_kb=win_kb,
      out=f"{wd}/miami_{label}_{region}_{sexes[0]}_vs_{sexes[1]}_nominal.png",
      squeeze_gaps=True)